- 3D 虚拟引擎
    - Unreal Engine，Unity
- misc
    - BEV：Bird's Eye View
    - SLAM: Simultaneous Localization and Mapping（同步定位与建图）的缩写。

### Entity

- 锚点
    - 注意区别锚点和几何中心，当设置 Position = (10, 0, 0) 时，你其实是把物体的锚点放在了世界坐标的 (10, 0, 0) 处，而不是物体的几何中心。
- relative aabb vs. world aabb
    - 碰撞检测走的是 world aabb (必须永远和世界坐标轴（X, Y, Z）对齐，)
    - relative aabb 存在的意义在于降低矩阵运算的计算量
        - relative aabb 提供了 8 个角，物体旋转时，CPU 只需要对这 8 个角 进行矩阵变换。物体旋转时，CPU 只需要对这 8 个角 进行矩阵变换。算出这 8 个新点的位置，取最大最小值，就立刻得到了新的 World AABB。
- world AABB footprint
    - 指该 3D 包围盒在二维平面（通常指地表，如 XZ 平面）上的正投影面积或三维占地空间。可以将其具象化理解为一栋建筑物或一个角色在地表投射下的绝对几何地基轮廓。
- aabb vs. obb
    - https://gemini.google.com/share/ffa918d201a6
    - **AABB（Axis-Aligned Bounding Box，轴对齐包围盒）**与 **OBB（Oriented Bounding Box，有向包围盒）**
    - 观察当物体旋转时，AABB 为了保持坐标轴对齐，其体积会剧烈膨胀，导致在视觉上未接触时就引发“误判”（即宽相检测为正，常需要进入窄相二次确认）。而 OBB 如影随形，紧密贴合几何体。但代价是，系统需在底层执行高达 15 次的分离轴（SAT）投影测试。

### 欧拉角和四元素

> 轴-角（Axis-Angle）

- https://www.youtube.com/watch?v=zjMuIxRvygQ
- 欧拉角（Euler Angles）是**“三次连续的旋转”，而四元数（Quaternion）是“绕任意一根轴的一次旋转”**。
    - 欧拉证明过：任何 3D 旋转都可以简化为“绕空间中某一个特定的轴（Axis），旋转特定的角度（Angle）”。
    - 四元数 $(w, x, y, z)$ 本质上就是这个**“轴+角”**的高级数学压缩包：
        - 轴 (Axis)：空间中的一根矢量箭头 $\vec{v}$。
        - 角 (Angle)：绕这根箭头的旋转量 $\theta$。
$$
\begin{aligned}
w &= \cos(\frac{\theta}{2}) \\
\vec{xyz} &= \vec{v} \cdot \sin(\frac{\theta}{2})
\end{aligned}
$$

### 硬件

- IMU：Inertial Measurement Unit（惯性测量单元）
    - 加速度计 (Accelerometer)：
    - 陀螺仪 (Gyroscope)：
    - 磁力计 (Magnetometer)：
- IMU 在机器人中起什么作用？
    - 姿态平衡 (Balance & Stability)
    - 定位与导航 (Localization & Navigation)

### RGB-D

- RGB-D 相机
    - RGB：HxWx3
    - Depth: 提供几何信息（H × W × 1）。深度图通常是一张灰度图，像素值代表该点距离相机的真实物理距离（通常单位是毫米或米）。
- 理解 RGBD 相机的关键在于掌握如何将一个像素点 $(u, v)$ 转换到三维空间中的坐标 $(x, y, z)$。这需要用到相机的内参矩阵 (Intrinsic Matrix)：
    - 通过相机内参，将深度像素反投影为 3D 点云。
        - 无数个离散的点在三维坐标系中组成的集合。如果说 2D 图像是由像素（Pixel）组成的矩阵，那么 3D 模型就是由**点（Point）**组成的云团。
    $$
    K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}
    $$
  $$
  Z \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} X \\ Y \\ Z \end{bmatrix}
  $$
    - 如果你已知像素坐标 $(u, v)$ 和该点的深度值 $d$，你可以通过以下公式计算其在相机坐标系下的 3D 位置：
        - $z = d$
        - $x = \frac{(u - c_x) \cdot z}{f_x}$
        - $y = \frac{(v - c_y) \cdot z}{f_y}$
    - 将图像中所有的像素点都进行这样的转换，就构成了我们常说的点云 (Point Cloud)。

### 点云（point cloud）与体素（Voxel）


https://gemini.google.com/app/bb854bfc7c38eb2b

- 体素 (Voxel) 这个词是 Volume (体积) 和 element (元素) / Pixel (像素) 的组合。
- 体素化 (Voxelization)：点云通常是原始数据，数据量巨大且杂乱无章。为了让计算机更容易理解（例如判断碰撞、计算路径），我们通常会将点云体素化。
- 点云 & 体素
    - 点云是离散的点集合（$\{x,y,z\}$），它是无序的，放大看只有空隙。
    - 将空间划分为网格（$\text{Grid}[x][y][z]$）。如果网格内有点，则填充一个立方体。这是 3D 数据的“数字化/像素化”形态，是有序且结构化的。

### Occupancy Map & Frontier

- Frontier 的数学定义是**“已知自由空间”（Free Space）与“未知空间”（Unknown Space）的交界处**。要计算它，你只需要构建一个占据栅格地图（Occupancy Grid Map）或体素地图（Voxel Map）。
- Occupancy Map（占用栅格地图）
    - 3D 空间划分为无数个小的立方体单元（称为 体素/Voxel），并为每个单元分配一个状态或概率值，用来描述该空间位置是 “被占据的” (Occupied)、“空闲的” (Free) 还是 “未知的” (Unknown)。
    - Occupancy Map 是 SLAM 的建图结果